# Music Genre Classifier

**CS 89.02 / MUS 14.05 — Music and AI, Week 4**

This notebook demonstrates building a music genre classifier using two approaches:

1. **SVM with hand-crafted features** — extract MFCCs, chroma, and spectral features,
   then train a Support Vector Machine.
2. **Simple CNN on mel spectrograms** — feed spectrogram images directly into a
   convolutional neural network.

We use a synthetic dataset for demonstration. For the assignment, you should use
real audio data (GTZAN, FMA, or your own collection).

In [ ]:
!pip install librosa scikit-learn matplotlib torch torchaudio

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')


def extract_features(y, sr):
    """Extract a feature vector from an audio signal.

    Features: 13 MFCCs (mean+std), 12 chroma (mean+std),
    spectral centroid, bandwidth, rolloff, ZCR (mean+std each),
    tempo, RMS (mean+std).
    """
    feat = []

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    feat.extend(np.mean(mfccs, axis=1))
    feat.extend(np.std(mfccs, axis=1))

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    feat.extend(np.mean(chroma, axis=1))
    feat.extend(np.std(chroma, axis=1))

    # Spectral centroid
    sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    feat.extend([np.mean(sc), np.std(sc)])

    # Spectral bandwidth
    sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    feat.extend([np.mean(sb), np.std(sb)])

    # Spectral rolloff
    sro = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    feat.extend([np.mean(sro), np.std(sro)])

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    feat.extend([np.mean(zcr), np.std(zcr)])

    # Tempo — librosa 0.10+ returns a 1-element numpy array
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    feat.append(float(tempo[0]))

    # RMS energy
    rms = librosa.feature.rms(y=y)[0]
    feat.extend([np.mean(rms), np.std(rms)])

    return np.array(feat)


print(f"Feature vector length: {len(extract_features(np.random.randn(22050), 22050))}")

## Dataset

For this demo we create a **synthetic dataset** by manipulating librosa's built-in
audio examples with different transformations to simulate genre differences.

In practice, you would use a real dataset such as:
- **GTZAN** — 1000 clips, 10 genres, 30 sec each (the "MNIST of music")
- **FMA** — Free Music Archive, various sizes (small/medium/large)
- **Your own collection** — at least 10 clips per class

We simulate three "genres" by pitch-shifting, time-stretching, and adding
noise/filtering to the same source audio.

In [ ]:
# Load base audio
y_base, sr = librosa.load(librosa.example('brahms'), duration=30)

def make_synthetic_dataset(y, sr, n_per_class=15, seed=42):
    """Create a synthetic dataset with 3 'genre' classes.

    Class 0 ('orchestral'): original with slight variations
    Class 1 ('bright'): pitch-shifted up, brighter spectrum
    Class 2 ('dark'): pitch-shifted down, low-pass filtered
    """
    rng = np.random.RandomState(seed)
    X, labels = [], []
    clip_len = 3 * sr  # 3-second clips

    for i in range(n_per_class):
        # Random starting position
        start = rng.randint(0, len(y) - clip_len)
        clip = y[start:start + clip_len]

        # Class 0: 'orchestral' — original with slight noise
        c0 = clip + rng.randn(len(clip)) * 0.005
        X.append(extract_features(c0, sr))
        labels.append(0)

        # Class 1: 'bright' — pitch up + emphasis on highs
        c1 = librosa.effects.pitch_shift(clip, sr=sr, n_steps=rng.uniform(3, 6))
        X.append(extract_features(c1, sr))
        labels.append(1)

        # Class 2: 'dark' — pitch down + low energy
        c2 = librosa.effects.pitch_shift(clip, sr=sr, n_steps=rng.uniform(-6, -3))
        c2 = c2 * 0.5  # reduce volume
        X.append(extract_features(c2, sr))
        labels.append(2)

    return np.array(X), np.array(labels)


X, y_labels = make_synthetic_dataset(y_base, sr)
genre_names = ['orchestral', 'bright', 'dark']

print(f"Dataset shape: {X.shape}")
print(f"Classes: {np.bincount(y_labels)} samples per class")
print(f"Feature vector length: {X.shape[1]}")

## SVM Classifier

Support Vector Machines find a hyperplane that maximally separates classes in
feature space. With an RBF kernel, SVMs can learn nonlinear boundaries.

We use a **Pipeline** that first standardizes features (zero mean, unit variance)
then applies the SVM. This prevents data leakage during cross-validation.

In [ ]:
# Build SVM pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', random_state=42))
])

# Stratified 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_scores = cross_val_score(svm_pipeline, X, y_labels, cv=cv, scoring='accuracy')

print(f"SVM Cross-Validation Accuracy: {svm_scores.mean():.3f} +/- {svm_scores.std():.3f}")
print(f"Per-fold scores: {svm_scores}")

# Train on full data for confusion matrix visualization
svm_pipeline.fit(X, y_labels)
y_pred_svm = svm_pipeline.predict(X)

# Confusion matrix
cm = confusion_matrix(y_labels, y_pred_svm)
disp = ConfusionMatrixDisplay(cm, display_labels=genre_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues')
ax.set_title('SVM Confusion Matrix (training set)')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_labels, y_pred_svm, target_names=genre_names))

## Simple CNN Classifier

Instead of hand-crafted features, we can feed **mel spectrograms** directly
into a Convolutional Neural Network (CNN). The CNN learns its own features
from the spectrogram images.

This is a minimal 1D CNN for demonstration. State-of-the-art systems use
2D CNNs (treating spectrograms as images) or pretrained audio models.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


def extract_mel_spectrogram(y, sr, n_mels=64, max_len=128):
    """Extract a fixed-size mel spectrogram for CNN input."""
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    S_dB = librosa.power_to_db(S, ref=np.max)
    # Pad or truncate to fixed length
    if S_dB.shape[1] < max_len:
        S_dB = np.pad(S_dB, ((0, 0), (0, max_len - S_dB.shape[1])))
    else:
        S_dB = S_dB[:, :max_len]
    return S_dB


class GenreCNN(nn.Module):
    """Simple 1D CNN for mel spectrogram classification."""

    def __init__(self, n_mels=64, n_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(n_mels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, n_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        return self.fc(x)


# Build spectrogram dataset
y_base_full, sr = librosa.load(librosa.example('brahms'), duration=30)
rng = np.random.RandomState(42)
clip_len = 3 * sr
specs, spec_labels = [], []

for i in range(15):
    start = rng.randint(0, len(y_base_full) - clip_len)
    clip = y_base_full[start:start + clip_len]

    # Class 0
    c0 = clip + rng.randn(len(clip)) * 0.005
    specs.append(extract_mel_spectrogram(c0, sr))
    spec_labels.append(0)

    # Class 1
    c1 = librosa.effects.pitch_shift(clip, sr=sr, n_steps=rng.uniform(3, 6))
    specs.append(extract_mel_spectrogram(c1, sr))
    spec_labels.append(1)

    # Class 2
    c2 = librosa.effects.pitch_shift(clip, sr=sr, n_steps=rng.uniform(-6, -3)) * 0.5
    specs.append(extract_mel_spectrogram(c2, sr))
    spec_labels.append(2)

X_spec = torch.FloatTensor(np.array(specs))
y_spec = torch.LongTensor(spec_labels)

print(f"Spectrogram tensor shape: {X_spec.shape}  (samples x n_mels x time)")

# Train CNN
model = GenreCNN(n_mels=64, n_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

dataset = TensorDataset(X_spec, y_spec)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

losses = []
model.train()
for epoch in range(50):
    epoch_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))

# Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Training Loss')
plt.tight_layout()
plt.show()

# Evaluate
model.eval()
with torch.no_grad():
    preds = model(X_spec).argmax(dim=1).numpy()

cnn_acc = (preds == y_spec.numpy()).mean()
print(f"\nCNN Training Accuracy: {cnn_acc:.3f}")

## Comparison and Analysis

Let us compare the two approaches and examine which features contribute most
to the SVM classifier's decisions.

In [ ]:
# Compare accuracies
print("=" * 50)
print("Model Comparison")
print("=" * 50)
print(f"SVM (5-fold CV):    {svm_scores.mean():.3f} +/- {svm_scores.std():.3f}")
print(f"CNN (training set): {cnn_acc:.3f}")
print()
print("Note: The CNN accuracy is on training data (no CV here for brevity).")
print("For a fair comparison, you should use CV for both.")

# Feature importance via SVM coefficient magnitudes
# (Re-train a linear SVM for interpretability)
svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=10, random_state=42))
])
svm_linear.fit(X, y_labels)

# For multiclass, coef_ has shape (n_classes * (n_classes-1) / 2, n_features)
coef_magnitudes = np.abs(svm_linear.named_steps['svm'].coef_).mean(axis=0)

# Feature names
feature_names = (
    [f'mfcc_{i}_mean' for i in range(13)] +
    [f'mfcc_{i}_std' for i in range(13)] +
    [f'chroma_{i}_mean' for i in range(12)] +
    [f'chroma_{i}_std' for i in range(12)] +
    ['spec_centroid_mean', 'spec_centroid_std',
     'spec_bw_mean', 'spec_bw_std',
     'spec_rolloff_mean', 'spec_rolloff_std',
     'zcr_mean', 'zcr_std',
     'tempo', 'rms_mean', 'rms_std']
)

# Plot top 15 features
top_idx = np.argsort(coef_magnitudes)[-15:]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(15), coef_magnitudes[top_idx])
ax.set_yticks(range(15))
ax.set_yticklabels([feature_names[i] for i in top_idx])
ax.set_xlabel('Mean |coefficient|')
ax.set_title('Top 15 Features by SVM Coefficient Magnitude (Linear Kernel)')
plt.tight_layout()
plt.show()